# JagaDia - 01. Exploratory Data Analysis & Baseline Model

Notebook ini bertujuan untuk:
1. Memuat dataset kriminalitas dan data lingkungan (*spatial & temporal*).
2. Melakukan analisis distribusi spasial (hotspot) dan temporal (jam rawan).
3. Membangun model baseline (Random Forest & XGBoost) untuk klasifikasi risiko kerawanan jalan.

In [ ]:
import sys
from pathlib import Path

# Tambahkan direktori root ML ke path
ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.append(str(ml_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, RANDOM_STATE
from src.data_pipeline import generate_mock_crime_dataset, load_data
from src.features import extract_temporal_features

print("Setup completed successfully.")

## 1. Load Data
Jika data riil belum tersedia, kita dapat men-generate data uji realistis menggunakan modul `data_pipeline`.

In [ ]:
raw_csv = RAW_DATA_DIR / "sample_crime_data.csv"
if not raw_csv.exists():
    df = generate_mock_crime_dataset(n_samples=2000, output_path=raw_csv)
else:
    df = load_data(raw_csv)

df.head()

## 2. Analisis Temporal (Waktu Kejadian)
Menganalisis pola kejahatan berdasarkan jam (apakah terpusat pada 18.00 - 06.00).

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(data=df, x='hour', hue='is_high_risk', palette='coolwarm')
plt.title("Distribusi Tingkat Kerawanan Berdasarkan Jam")
plt.xlabel("Jam (0 - 23)")
plt.ylabel("Jumlah Kasus/Sampel")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 3. Analisis Faktor Lingkungan (Penerangan & Pos Polisi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(ax=axes[0], data=df, x='is_high_risk', y='street_light_density', palette='Set2')
axes[0].set_title("Kepadatan Penerangan vs Kerawanan")

sns.boxplot(ax=axes[1], data=df, x='is_high_risk', y='dist_to_police_station', palette='Set2')
axes[1].set_title("Jarak Pos Polisi (m) vs Kerawanan")

plt.tight_layout()
plt.show()

## 4. Run Baseline Model
Kita memanggil fungsi training baseline yang telah dibuat di modul `src.train_baseline`.

In [ ]:
from src.train_baseline import train_baseline

train_baseline()